In [1]:
!pip install google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client

  Using cached google_auth_oauthlib-1.3.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached oauthlib-3.3.1-py3-none-any.whl.metadata (7.9 kB)
Using cached google_auth_oauthlib-1.3.0-py3-none-any.whl (19 kB)
Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl (24 kB)
Using cached oauthlib-3.3.1-py3-none-any.whl (160 kB)

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import base64
import json
import random
import string
import google.generativeai as genai
from datetime import datetime, timedelta, timezone, UTC
from dotenv import load_dotenv
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# Google API
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build

/home/robotica/Documentos/IA-minirobots/S7_IA_generativa/Trabajo/IA_minibots_T7_IA_generativa/Gemini/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_29000/66986399.py:6: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [5]:

SCOPES = [
    'https://www.googleapis.com/auth/gmail.readonly',
    'https://www.googleapis.com/auth/gmail.send',
    'https://www.googleapis.com/auth/calendar',
]

# Ruta a credenciales (credentials.json debe estar en el mismo directorio)
BASE_DIR = os.path.dirname(os.path.abspath('03_agente_personal.ipynb'))
CREDENTIALS_PATH = os.path.join(BASE_DIR, 'credentials.json')
TOKEN_PATH = os.path.join(BASE_DIR, 'token.json')


def get_google_services():
    """Autentica con OAuth2 y retorna los servicios de Gmail y Calendar."""
    creds = None

    if os.path.exists(TOKEN_PATH):
        creds = Credentials.from_authorized_user_file(TOKEN_PATH, SCOPES)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            if not os.path.exists(CREDENTIALS_PATH):
                raise FileNotFoundError(
                    f"No se encontró '{CREDENTIALS_PATH}'.\n"
                    "Descarga las credenciales OAuth2 Desktop desde Google Cloud Console "
                    "y coloca el archivo como 'credentials.json' en el mismo directorio."
                )
            flow = InstalledAppFlow.from_client_secrets_file(CREDENTIALS_PATH, SCOPES)
            creds = flow.run_local_server(port=0)

        with open(TOKEN_PATH, 'w') as token:
            token.write(creds.to_json())

    gmail_service = build('gmail', 'v1', credentials=creds)
    calendar_service = build('calendar', 'v3', credentials=creds)
    return gmail_service, calendar_service


In [7]:
_borrador_correo: dict = {}          # Almacena el correo pendiente de confirmación
_clave_confirmacion: str = ""        # Clave generada para confirmar el envío
_gmail_service = None
_calendar_service = None


def _genai_mini(prompt: str) -> str:
    """Helper: usa gemini-2.5-flash-lite para tareas internas de resumen."""
    temp_model = genai.GenerativeModel('gemini-2.5-flash-lite')
    return temp_model.generate_content(prompt).text.strip()


In [8]:
#Tools
def listar_correos(max_resultados: int = 10, solo_no_leidos: bool = False) -> str:
    """Lista los correos más recientes de la bandeja de entrada.

    Args:
        max_resultados: Número máximo de correos a mostrar (por defecto 10, máximo 25).
        solo_no_leidos: Si es True, muestra únicamente correos no leídos.
    """
    print(f"[Tool] Listando correos (max={max_resultados}, no_leidos={solo_no_leidos})")
    try:
        query = "in:inbox is:unread" if solo_no_leidos else "in:inbox"
        max_resultados = min(max_resultados, 25)
        result = _gmail_service.users().messages().list(
            userId='me', q=query, maxResults=max_resultados
        ).execute()

        messages = result.get('messages', [])
        if not messages:
            return "No se encontraron correos."

        output = []
        for msg in messages:
            detail = _gmail_service.users().messages().get(
                userId='me', id=msg['id'], format='metadata',
                metadataHeaders=['From', 'Subject', 'Date']
            ).execute()
            headers = {h['name']: h['value'] for h in detail['payload']['headers']}
            snippet = detail.get('snippet', '')[:100]
            labels = detail.get('labelIds', [])
            leido = "✉ No leído" if 'UNREAD' in labels else "✔ Leído"
            output.append(
                f"ID: {msg['id']}\n"
                f"  De: {headers.get('From', 'Desconocido')}\n"
                f"  Asunto: {headers.get('Subject', '(sin asunto)')}\n"
                f"  Fecha: {headers.get('Date', '')}\n"
                f"  Estado: {leido}\n"
                f"  Extracto: {snippet}..."
            )
        return "\n\n".join(output)
    except Exception as e:
        return f"Error al listar correos: {e}"


def buscar_correos(consulta: str, max_resultados: int = 5) -> str:
    """Busca correos en Gmail usando una consulta de búsqueda estilo Gmail.

    Ejemplos de consultas:
       - "from:pedro@email.com"
       - "subject:factura after:2025/01/01"
       - "has:attachment is:unread"

    Args:
        consulta: Cadena de búsqueda compatible con el buscador de Gmail.
        max_resultados: Número máximo de resultados (máximo 20).
    """
    print(f"[Tool] Buscando correos con: '{consulta}'")
    try:
        max_resultados = min(max_resultados, 20)
        result = _gmail_service.users().messages().list(
            userId='me', q=consulta, maxResults=max_resultados
        ).execute()

        messages = result.get('messages', [])
        if not messages:
            return f"No se encontraron correos para la búsqueda: '{consulta}'"

        output = []
        for msg in messages:
            detail = _gmail_service.users().messages().get(
                userId='me', id=msg['id'], format='metadata',
                metadataHeaders=['From', 'Subject', 'Date']
            ).execute()
            headers = {h['name']: h['value'] for h in detail['payload']['headers']}
            snippet = detail.get('snippet', '')[:120]
            output.append(
                f"ID: {msg['id']}\n"
                f"  De: {headers.get('From', 'Desconocido')}\n"
                f"  Asunto: {headers.get('Subject', '(sin asunto)')}\n"
                f"  Fecha: {headers.get('Date', '')}\n"
                f"  Extracto: {snippet}..."
            )
        return f"Resultados para '{consulta}':\n\n" + "\n\n".join(output)
    except Exception as e:
        return f"Error en la búsqueda de correos: {e}"


def leer_correo_completo(correo_id: str) -> str:
    """Lee el contenido completo de un correo específico usando su ID.

    Args:
        correo_id: El ID del correo a leer (obtenido con listar_correos o buscar_correos).
    """
    print(f"[Tool] Leyendo correo: {correo_id}")
    try:
        msg = _gmail_service.users().messages().get(
            userId='me', id=correo_id, format='full'
        ).execute()

        headers = {h['name']: h['value'] for h in msg['payload']['headers']}
        body = ""

        def extract_body(payload):
            """Extrae el texto del cuerpo del correo recursivamente."""
            if 'parts' in payload:
                for part in payload['parts']:
                    result = extract_body(part)
                    if result:
                        return result
            elif payload.get('mimeType') == 'text/plain':
                data = payload.get('body', {}).get('data', '')
                if data:
                    return base64.urlsafe_b64decode(data).decode('utf-8', errors='replace')
            return ""

        body = extract_body(msg['payload'])
        if not body:
            body = base64.urlsafe_b64decode(
                msg['payload'].get('body', {}).get('data', b'')
            ).decode('utf-8', errors='replace')

        return (
            f"De: {headers.get('From', 'Desconocido')}\n"
            f"Para: {headers.get('To', '')}\n"
            f"Asunto: {headers.get('Subject', '(sin asunto)')}\n"
            f"Fecha: {headers.get('Date', '')}\n"
            f"{'─' * 50}\n"
            f"{body[:3000]}"
        )
    except Exception as e:
        return f"Error al leer correo: {e}"


def resumir_correos(max_correos: int = 5) -> str:
    """Obtiene los últimos correos de la bandeja de entrada y genera un resumen ejecutivo con IA.

    Args:
        max_correos: Número de correos a resumir (entre 1 y 15).
    """
    print(f"[Tool] Generando resumen de los últimos {max_correos} correos")
    try:
        max_correos = min(max(max_correos, 1), 15)
        result = _gmail_service.users().messages().list(
            userId='me', q='in:inbox', maxResults=max_correos
        ).execute()

        messages = result.get('messages', [])
        if not messages:
            return "No hay correos en la bandeja de entrada."

        textos = []
        for msg in messages:
            detail = _gmail_service.users().messages().get(
                userId='me', id=msg['id'], format='metadata',
                metadataHeaders=['From', 'Subject', 'Date']
            ).execute()
            headers = {h['name']: h['value'] for h in detail['payload']['headers']}
            snippet = detail.get('snippet', '')
            textos.append(
                f"- De: {headers.get('From', '?')} | Asunto: {headers.get('Subject', '?')} | "
                f"Fecha: {headers.get('Date', '?')} | Extracto: {snippet}"
            )

        texto_correos = "\n".join(textos)
        resumen = _genai_mini(
            f"Eres un asistente ejecutivo. Resume de forma concisa y clara los siguientes "
            f"{max_correos} correos recientes. Identifica correos urgentes, solicitudes pendientes "
            f"y cualquier acción requerida:\n\n{texto_correos}"
        )
        return f"Resumen de los últimos {len(messages)} correos:\n\n{resumen}"
    except Exception as e:
        return f"Error al resumir correos: {e}"


def preparar_correo(destinatario: str, asunto: str, cuerpo: str, cc: str = "") -> str:
    """Prepara un correo para enviar. Genera una clave de confirmación que el usuario
    debe proporcionar para autorizar el envío real. NO envía el correo todavía.

    Args:
        destinatario: Dirección de correo del destinatario principal.
        asunto: Asunto del correo.
        cuerpo: Contenido del mensaje en texto plano.
        cc: Direcciones en copia (opcional, separadas por comas).
    """
    global _borrador_correo, _clave_confirmacion
    print(f"[Tool] Preparando correo para: {destinatario} | Asunto: {asunto}")

    _clave_confirmacion = ''.join(random.choices(string.ascii_uppercase + string.digits, k=6))
    _borrador_correo = {
        'destinatario': destinatario,
        'asunto': asunto,
        'cuerpo': cuerpo,
        'cc': cc
    }

    preview = (
        f"Correo listo para enviar:\n"
        f"  Para: {destinatario}\n"
        + (f"  CC: {cc}\n" if cc else "")
        + f"  Asunto: {asunto}\n"
        f"  Cuerpo:\n  {'─' * 40}\n  {cuerpo[:300]}{'...' if len(cuerpo) > 300 else ''}\n"
        f"  {'─' * 40}\n\n"
        f"Clave de confirmación: **{_clave_confirmacion}**\n"
        f"Para confirmar el envío, dime: 'Envía con clave {_clave_confirmacion}'"
    )
    return preview


def confirmar_y_enviar_correo(clave_confirmacion: str) -> str:
    """Envía el correo previamente preparado si la clave de confirmación es correcta.

    Args:
        clave_confirmacion: La clave de 6 caracteres generada por preparar_correo.
    """
    global _borrador_correo, _clave_confirmacion
    print(f"[Tool] Intentando enviar correo con clave: {clave_confirmacion}")

    if not _borrador_correo:
        return "No hay ningún correo preparado. Usa preparar_correo() primero."

    if clave_confirmacion.strip().upper() != _clave_confirmacion:
        intentos_msg = f"Clave incorrecta ('{clave_confirmacion}'). La clave esperada tiene 6 caracteres. El correo NO fue enviado."
        return intentos_msg

    try:
        msg = MIMEMultipart()
        msg['To'] = _borrador_correo['destinatario']
        msg['Subject'] = _borrador_correo['asunto']
        if _borrador_correo.get('cc'):
            msg['Cc'] = _borrador_correo['cc']
        msg.attach(MIMEText(_borrador_correo['cuerpo'], 'plain', 'utf-8'))

        raw = base64.urlsafe_b64encode(msg.as_bytes()).decode()
        _gmail_service.users().messages().send(
            userId='me', body={'raw': raw}
        ).execute()

        destinatario = _borrador_correo['destinatario']
        asunto = _borrador_correo['asunto']
        _borrador_correo = {}
        _clave_confirmacion = ""
        return f"Correo enviado correctamente a **{destinatario}** con asunto **'{asunto}'**."
    except Exception as e:
        return f"Error al enviar el correo: {e}"


def cancelar_correo_pendiente() -> str:
    """Cancela el correo que estaba pendiente de confirmación, descartando el borrador."""
    global _borrador_correo, _clave_confirmacion
    if not _borrador_correo:
        return "No hay ningún correo pendiente de envío."
    _borrador_correo = {}
    _clave_confirmacion = ""
    return "Correo cancelado. El borrador ha sido descartado."


# ══════════════════════════════════════════════
# HERRAMIENTAS GOOGLE CALENDAR
# ══════════════════════════════════════════════

def listar_eventos(dias_adelante: int = 7) -> str:
    """Lista los próximos eventos del calendario principal de Google Calendar.

    Args:
        dias_adelante: Número de días hacia adelante a consultar (por defecto 7, máximo 60).
    """
    print(f"[Tool] Listando eventos de los próximos {dias_adelante} días")
    try:
        dias_adelante = min(max(dias_adelante, 1), 60)
        now = datetime.now(UTC)
        time_max = now + timedelta(days=dias_adelante)

        events_result = _calendar_service.events().list(
            calendarId='primary',
            timeMin=now.isoformat(),
            timeMax=time_max.isoformat(),
            maxResults=30,
            singleEvents=True,
            orderBy='startTime'
        ).execute()

        events = events_result.get('items', [])
        if not events:
            return f"No hay eventos en los próximos {dias_adelante} días."

        output = []
        for ev in events:
            start = ev['start'].get('dateTime', ev['start'].get('date', ''))
            end = ev['end'].get('dateTime', ev['end'].get('date', ''))
            desc = ev.get('description', '')
            location = ev.get('location', '')
            output.append(
                f"{ev.get('summary', '(sin título)')}\n"
                f"   ID: {ev['id']}\n"
                f"   Inicio: {start}\n"
                f"   Fin: {end}\n"
                + (f"   Lugar: {location}\n" if location else "")
                + (f"   Descripción: {desc[:100]}\n" if desc else "")
            )
        return f"Eventos en los próximos {dias_adelante} días:\n\n" + "\n".join(output)
    except Exception as e:
        return f"Error al listar eventos: {e}"


def crear_evento(
    titulo: str,
    fecha_inicio: str,
    fecha_fin: str,
    descripcion: str = "",
    lugar: str = "",
    recordatorio_minutos: int = 30
) -> str:
    """Crea un nuevo evento en Google Calendar.

    Args:
        titulo: Nombre o título del evento.
        fecha_inicio: Fecha y hora de inicio en formato ISO 8601 (ej: '2025-04-15T10:00:00-05:00').
        fecha_fin: Fecha y hora de fin en formato ISO 8601 (ej: '2025-04-15T11:00:00-05:00').
        descripcion: Descripción opcional del evento.
        lugar: Lugar o dirección opcional del evento.
        recordatorio_minutos: Minutos de antelación para el recordatorio (por defecto 30).
    """
    print(f"[Tool] Creando evento: '{titulo}' ({fecha_inicio} → {fecha_fin})")
    try:
        event = {
            'summary': titulo,
            'start': {'dateTime': fecha_inicio},
            'end': {'dateTime': fecha_fin},
            'reminders': {
                'useDefault': False,
                'overrides': [
                    {'method': 'popup', 'minutes': recordatorio_minutos},
                    {'method': 'email', 'minutes': recordatorio_minutos},
                ]
            }
        }
        if descripcion:
            event['description'] = descripcion
        if lugar:
            event['location'] = lugar

        created = _calendar_service.events().insert(
            calendarId='primary', body=event
        ).execute()

        return (
            f"✅ Evento creado exitosamente:\n"
            f"   Título: {titulo}\n"
            f"   Inicio: {fecha_inicio}\n"
            f"   Fin: {fecha_fin}\n"
            f"   ID: {created['id']}\n"
            f"   Enlace: {created.get('htmlLink', 'N/A')}"
        )
    except Exception as e:
        return f"Error al crear el evento: {e}"


def modificar_evento(
    evento_id: str,
    nuevo_titulo: str = "",
    nueva_fecha_inicio: str = "",
    nueva_fecha_fin: str = "",
    nueva_descripcion: str = ""
) -> str:
    """Modifica un evento existente en Google Calendar.

    Args:
        evento_id: ID del evento a modificar (obtenido con listar_eventos).
        nuevo_titulo: Nuevo título del evento (dejar vacío para no cambiar).
        nueva_fecha_inicio: Nueva fecha/hora de inicio en ISO 8601 (dejar vacío para no cambiar).
        nueva_fecha_fin: Nueva fecha/hora de fin en ISO 8601 (dejar vacío para no cambiar).
        nueva_descripcion: Nueva descripción (dejar vacío para no cambiar).
    """
    print(f"[Tool] Modificando evento: {evento_id}")
    try:
        event = _calendar_service.events().get(
            calendarId='primary', eventId=evento_id
        ).execute()

        if nuevo_titulo:
            event['summary'] = nuevo_titulo
        if nueva_fecha_inicio:
            event['start'] = {'dateTime': nueva_fecha_inicio}
        if nueva_fecha_fin:
            event['end'] = {'dateTime': nueva_fecha_fin}
        if nueva_descripcion:
            event['description'] = nueva_descripcion

        updated = _calendar_service.events().update(
            calendarId='primary', eventId=evento_id, body=event
        ).execute()

        return (
            f"Evento actualizado:\n"
            f"   Título: {updated.get('summary')}\n"
            f"   Inicio: {updated['start'].get('dateTime', updated['start'].get('date'))}\n"
            f"   ID: {evento_id}"
        )
    except Exception as e:
        return f"Error al modificar el evento: {e}"


def eliminar_evento(evento_id: str) -> str:
    """Elimina un evento del calendario usando su ID.

    Args:
        evento_id: El ID del evento a eliminar (obtenido con listar_eventos).
    """
    print(f"[Tool] Eliminando evento: {evento_id}")
    try:
        _calendar_service.events().delete(
            calendarId='primary', eventId=evento_id
        ).execute()
        return f"Evento con ID '{evento_id}' eliminado correctamente."
    except Exception as e:
        return f"Error al eliminar el evento: {e}"


def resumen_tareas_pendientes() -> str:
    """Genera un resumen ejecutivo de todas las tareas y eventos pendientes en los
    próximos 7 días usando IA. Ideal para planificación diaria o semanal."""
    print("[Tool] Generando resumen de tareas pendientes del calendario")
    try:
        now = datetime.now(UTC)
        time_max = now + timedelta(days=7)

        events_result = _calendar_service.events().list(
            calendarId='primary',
            timeMin=now.isoformat(),
            timeMax=time_max.isoformat(),
            maxResults=50,
            singleEvents=True,
            orderBy='startTime'
        ).execute()

        events = events_result.get('items', [])
        if not events:
            return "No tienes eventos programados para los próximos 7 días. ¡Tu agenda está libre!"

        eventos_text = []
        for ev in events:
            start = ev['start'].get('dateTime', ev['start'].get('date', ''))
            desc = ev.get('description', '')
            eventos_text.append(
                f"- '{ev.get('summary', 'Sin título')}' el {start}"
                + (f": {desc[:80]}" if desc else "")
            )

        texto_eventos = "\n".join(eventos_text)
        hoy = now.strftime('%Y-%m-%d %H:%M')
        resumen = _genai_mini(
            f"Eres un asistente de productividad. La fecha y hora actual es {hoy}.\n"
            f"Aquí están los eventos de los próximos 7 días:\n\n{texto_eventos}\n\n"
            f"Genera un resumen ejecutivo claro, priorizando por urgencia e importancia. "
            f"Menciona qué eventos son hoy, mañana y el resto de la semana. "
            f"Finaliza con consejos breves de planificación si procede."
        )
        return f"Resumen de tareas pendientes (próximos 7 días):\n\n{resumen}"
    except Exception as e:
        return f"Error al generar el resumen de tareas: {e}"


def definir_utc(lugar: str) -> str:
    """Obtiene el offset de la zona horaria (UTC) de una ciudad, país o región.

    Args:
        lugar: Nombre del lugar (ej. "Bogotá", "Madrid", "Nueva York").
    """
    print(f'[Tool] Consultando UTC para: {lugar}')
    temp_model = genai.GenerativeModel('gemini-2.5-flash')
    respuesta = temp_model.generate_content(
        f"Devuelve ÚNICAMENTE el número que representa el offset UTC actual de {lugar} "
        f"(en horas). Ejemplo: si es UTC-5, responde solo '-5'."
    )
    offset = respuesta.text.strip()
    print(f'[Tool] UTC para {lugar}: {offset}')
    return offset


def get_current_time(timezone_offset: float) -> str:
    """Calcula la fecha y hora actual según un offset UTC.
    Si no conoces el offset para una ciudad, primero llama a definir_utc.

    Args:
        timezone_offset: Offset de la zona horaria en horas (ej: -5, 2, 5.5).
    """
    hora_utc = datetime.now(UTC)
    tz_local = timezone(timedelta(hours=timezone_offset))
    hora_local = hora_utc.astimezone(tz_local)

    hora_6pm = hora_local.replace(hour=18, minute=0, second=0, microsecond=0)
    if hora_local > hora_6pm:
        hora_6pm += timedelta(days=1)
    faltan = (hora_6pm - hora_local).total_seconds() / 3600

    res = (
        f"Hora UTC: {hora_utc.strftime('%Y-%m-%d %H:%M:%S')}, "
        f"Hora local: {hora_local.strftime('%Y-%m-%d %H:%M:%S')}, "
        f"Faltan {faltan:.2f} horas para las 18:00."
    )
    print(f'[Tool] Hora con offset {timezone_offset}: {res}')
    return res


In [9]:

class AgenteGmail:
    """Agente conversacional con acceso a Gmail y Google Calendar."""

    SYSTEM_PROMPT = """Eres un asistente personal inteligente y proactivo con acceso completo a Gmail y Google Calendar del usuario. Hoy es {fecha_hoy}.

Tus capacidades principales:
• **Correo**: listar, buscar, leer y enviar correos (con confirmación de seguridad).
• **Calendario**: ver eventos, crear, modificar y eliminar actividades, y generar resúmenes de tareas.
• **Tiempo**: consultar la hora actual en cualquier zona horaria.

Normas de comportamiento:
1. **Envío de correos**: Siempre usa `preparar_correo` primero. Nunca envíes sin que el usuario confirme la clave de seguridad.
2. **Proactividad**: Si el usuario pide planificar algo, sugiere horarios concretos y ofrece crear el evento directamente.
3. **Claridad**: Al listar correos o eventos, presenta la información de forma limpia y estructurada.
4. **Resúmenes inteligentes**: Cuando generes resúmenes, incluye siempre acciones recomendadas.
5. **Idioma**: Responde siempre en español a menos que el usuario indique lo contrario.
6. **Formato de fechas**: Cuando el usuario mencione fechas, convierte a ISO 8601 con la zona horaria correcta antes de crear eventos.

Cuando el usuario te saluda por primera vez, ofrece un resumen rápido del estado actual (correos no leídos y próximos eventos)."""

    def __init__(self, model_name: str = 'gemini-2.5-flash'):
        global _gmail_service, _calendar_service

        # Cargar variables de entorno
        load_dotenv()

        # API Key de Gemini
        api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("APIKEY")
        if not api_key:
            raise ValueError(
                "No se encontró la API KEY de Gemini. "
                "Configura GEMINI_API_KEY en tu .env o variable de entorno."
            )
        genai.configure(api_key=api_key)

        # Autenticación OAuth2 con Google
        print("Autenticando con Google (Gmail + Calendar)...")
        try:
            _gmail_service, _calendar_service = get_google_services()
            print("Autenticación exitosa.")
        except FileNotFoundError as e:
            print(f"\n{e}")
            print("\nEl agente funcionará sin Gmail/Calendar hasta que configures las credenciales.")
            _gmail_service = None
            _calendar_service = None

        # Herramientas
        self.herramientas = [
            listar_correos,
            buscar_correos,
            leer_correo_completo,
            resumir_correos,
            preparar_correo,
            confirmar_y_enviar_correo,
            cancelar_correo_pendiente,
            listar_eventos,
            crear_evento,
            modificar_evento,
            eliminar_evento,
            resumen_tareas_pendientes,
            get_current_time,
            definir_utc,
        ]

        fecha_hoy = datetime.now().strftime('%A, %d de %B de %Y, %H:%M')
        system_instruction = self.SYSTEM_PROMPT.format(fecha_hoy=fecha_hoy)

        self.model = genai.GenerativeModel(
            model_name=model_name,
            tools=self.herramientas,
            system_instruction=system_instruction,
        )

        self.chat = self.model.start_chat(enable_automatic_function_calling=True)

    def consultar(self, mensaje: str) -> str:
        """Envía un mensaje al agente y obtiene su respuesta."""
        try:
            respuesta = self.chat.send_message(mensaje)
            return respuesta.text
        except Exception as e:
            return f"Error al comunicarse con el modelo: {e}"


In [14]:
print("=" * 60)
print("Agente Gmail — Asistente Personal Inteligente")
print("=" * 60)
print("Escribe 'salir' o 'exit' para terminar la sesión.\n")

try:
    agente = AgenteGmail()
except Exception as e:
    print(f"\nError al iniciar el agente: {e}")
    exit()

print("\nAgente listo. ¿En qué te puedo ayudar hoy?\n")

while True:
    try:
        user_input = input("Tú: ").strip()
        if not user_input:
            continue
        if user_input.lower() in ('salir', 'exit', 'quit', 'q'):
            print("\n¡Hasta luego! Cerrando sesión.")
            break

        print("\nAgente: ", end="", flush=True)
        respuesta = agente.consultar(user_input)
        print(respuesta)
        print()

    except KeyboardInterrupt:
        print("\n\nSesión interrumpida. ¡Hasta luego!")
        break
    except Exception as e:
        print(f"\nError inesperado: {e}\n")

Agente Gmail — Asistente Personal Inteligente
Escribe 'salir' o 'exit' para terminar la sesión.

Autenticando con Google (Gmail + Calendar)...
Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=1066479574565-a4q1biepts128o93kvlg4j91v26ee7qo.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A51129%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.send+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcalendar&state=fgUU9HdOqpAEK0UiuKAQFozFJkJybT&code_challenge=wQxH49qEvlZJJi-2dnyL_Sflwcn8AruvE8KtUXWaVvQ&code_challenge_method=S256&access_type=offline
Abriendo en una sesión existente del navegador
Autenticación exitosa.

Agente listo. ¿En qué te puedo ayudar hoy?


Agente: Claro, puedo añadir "ir al medico" a tu calendario. ¿A qué hora sería la cita y a qué hora terminaría? Necesito la hora de inicio y fin para crear el evento.


Agente: [Tool] Lis